Cell 0 — Parameters (tag: parameters)

In [ ]:
# Cell 0 — tagged: parameters
# STEP files are read directly by gmsh via its OpenCASCADE kernel.
# This preserves exact geometry — curves and surfaces are not triangulated
# before meshing, giving significantly better mesh quality than
# converting to STL first.

import sys
sys.path.insert(0, "/workspace")

STEP_INPUT_PATH     = ""        # required: path to .step or .stp file
PART_NAME           = ""        # required: alphanumeric, underscores, hyphens only

# Mesh hints
TARGET_ELEMENT_SIZE = 2.0       # mm
MESH_ALGORITHM_3D   = 4         # 4=Frontal (best quality), 1=Delaunay (faster)

# Load hints
PRIMARY_FACE        = "top"
LOAD_MAGNITUDE_N    = 1000.0

# STEP-specific options
HEAL_SHAPES         = True      # run OpenCASCADE shape healing — fixes common
                                # STEP export artifacts from CAD tools
SCALE_FACTOR        = 1.0       # multiply all coordinates by this
                                # set to 1000.0 if your STEP is in metres

OUTPUT_DIR = "outputs"

Cell 1 — Validate inputs

In [ ]:
# Cell 1 — Validate STEP file exists and is a supported format
import re
from pathlib import Path

assert STEP_INPUT_PATH, "STEP_INPUT_PATH is required"
assert PART_NAME,       "PART_NAME is required"

assert re.match(r'^[a-zA-Z0-9_-]+$', PART_NAME), (
    f"PART_NAME '{PART_NAME}' contains invalid characters."
)

step_input = Path(STEP_INPUT_PATH)
assert step_input.exists(), f"File not found: {step_input}"
assert step_input.suffix.lower() in (".step", ".stp"), (
    f"Expected .step or .stp file, got: {step_input.suffix}. "
    "For STL files use 00_import_stl.ipynb instead."
)

file_size_kb = step_input.stat().st_size / 1024
print(f"STEP file:  {step_input}")
print(f"Size:       {file_size_kb:.1f} KB")
print(f"Part name:  {PART_NAME}")

Cell 2 — Load and inspect via gmsh OpenCASCADE kernel

In [ ]:
# Cell 2 — Import STEP into gmsh and inspect the geometry
#
# The OpenCASCADE kernel reads STEP exactly — faces, edges, and vertices
# are preserved as mathematical objects, not triangle approximations.
# This is why STEP → gmsh produces better meshes than STL → gmsh.

import gmsh
import numpy as np

if gmsh.is_initialized():
    gmsh.finalize()

gmsh.initialize()
gmsh.option.setNumber("General.Verbosity", 2)
gmsh.model.add(PART_NAME)

print("Loading STEP file via OpenCASCADE kernel...")
gmsh.model.occ.importShapes(str(step_input))

# Apply shape healing — fixes common problems from CAD export
# (short edges, small faces, gaps between surfaces)
if HEAL_SHAPES:
    gmsh.model.occ.healShapes()
    print("Shape healing applied")

# Apply scale factor if needed
if SCALE_FACTOR != 1.0:
    gmsh.model.occ.dilate(
        gmsh.model.occ.getEntities(),
        0, 0, 0,                          # origin
        SCALE_FACTOR, SCALE_FACTOR, SCALE_FACTOR,
    )
    print(f"Scaled by {SCALE_FACTOR}x")

gmsh.model.occ.synchronize()

# Report what gmsh found in the STEP file
volumes  = gmsh.model.getEntities(dim=3)
surfaces = gmsh.model.getEntities(dim=2)
curves   = gmsh.model.getEntities(dim=1)
points   = gmsh.model.getEntities(dim=0)

print(f"\nGeometry entities:")
print(f"  Volumes:   {len(volumes)}")
print(f"  Surfaces:  {len(surfaces)}")
print(f"  Curves:    {len(curves)}")
print(f"  Points:    {len(points)}")

if len(volumes) == 0:
    print("\n✗ No volumes found in STEP file.")
    print("  This usually means the STEP file contains surface geometry")
    print("  (shells or faces) rather than a solid body.")
    print("  Re-export from your CAD tool as a STEP solid (AP214 or AP203).")
elif len(volumes) > 1:
    print(f"\n⚠ {len(volumes)} volumes found — pipeline expects a single solid body.")
    print(f"  The largest volume will be used. Consider merging bodies in your")
    print(f"  CAD tool before exporting if this is unintentional.")

# Bounding box
bb = gmsh.model.getBoundingBox(-1, -1)
dims = np.array([bb[3]-bb[0], bb[4]-bb[1], bb[5]-bb[2]])
print(f"\nBounding box (after scaling):")
print(f"  X: [{bb[0]:.2f}, {bb[3]:.2f}]  →  {dims[0]:.2f} mm")
print(f"  Y: [{bb[1]:.2f}, {bb[4]:.2f}]  →  {dims[1]:.2f} mm")
print(f"  Z: [{bb[2]:.2f}, {bb[5]:.2f}]  →  {dims[2]:.2f} mm")

longest_dim = dims.max()
if longest_dim > 2000:
    print(f"\n⚠ Longest dimension {longest_dim:.1f} mm is very large.")
    print(f"  If your STEP file is in metres, set SCALE_FACTOR=1000.0 and re-run.")

Cell 3 — Tag physical groups

In [ ]:
# Cell 3 — Tag volume and surfaces as physical groups
#
# This is the same tagging logic as gmsh_pipeline.py but applied directly
# to the STEP geometry before meshing, rather than after STL import.
# Because STEP preserves surface topology, the classification is more
# reliable than the angle-based classifier used for STL imports.

# Tag all volumes as the bulk domain
vol_tags = [v[1] for v in volumes]
gmsh.model.addPhysicalGroup(3, vol_tags, tag=1)
gmsh.model.setPhysicalName(3, 1, "volume")
print("Tagged volume (physical group 1)")

# Classify surfaces by Z centroid
surface_centroids = []
for dim, tag in surfaces:
    bb_s = gmsh.model.getBoundingBox(dim, tag)
    z_center = (bb_s[2] + bb_s[5]) / 2
    surface_centroids.append((tag, z_center))

surface_centroids.sort(key=lambda x: x[1])

# Bottom surface — fixed BC in Stage 3
bot_tag = surface_centroids[0][0]
gmsh.model.addPhysicalGroup(2, [bot_tag], tag=3)
gmsh.model.setPhysicalName(2, 3, "bottom")
print(f"Tagged bottom surface (tag {bot_tag}) as physical group 3")

# Top surface — load application in Stage 3
top_tag = surface_centroids[-1][0]
gmsh.model.addPhysicalGroup(2, [top_tag], tag=2)
gmsh.model.setPhysicalName(2, 2, "top")
print(f"Tagged top surface (tag {top_tag}) as physical group 2")

# Side surfaces
side_tags = [s[0] for s in surface_centroids[1:-1]]
if side_tags:
    gmsh.model.addPhysicalGroup(2, side_tags, tag=4)
    gmsh.model.setPhysicalName(2, 4, "sides")
    print(f"Tagged {len(side_tags)} side surface(s) as physical group 4")

Cell 4 — Generate mesh and export

In [ ]:
# Cell 4 — Mesh the STEP geometry and export to XDMF
#
# Because we are working with exact OCC geometry, gmsh can apply
# mesh sizing directly to CAD surfaces and curves — no STL approximation.
# This typically gives better aspect ratios than the STL import path.

import meshio
from pathlib import Path

meshes_dir = Path(OUTPUT_DIR) / "meshes"
meshes_dir.mkdir(parents=True, exist_ok=True)

gmsh.option.setNumber("Mesh.Algorithm3D", MESH_ALGORITHM_3D)
gmsh.option.setNumber("Mesh.CharacteristicLengthMin", TARGET_ELEMENT_SIZE * 0.5)
gmsh.option.setNumber("Mesh.CharacteristicLengthMax", TARGET_ELEMENT_SIZE * 2.0)
gmsh.option.setNumber("Mesh.Optimize", 1)
gmsh.option.setNumber("Mesh.OptimizeNetgen", 1)

print(f"Generating mesh (element size ~{TARGET_ELEMENT_SIZE} mm)...")
gmsh.model.mesh.generate(3)

# Stats
node_tags, _, _ = gmsh.model.mesh.getNodes()
elem_types, elem_tags, _ = gmsh.model.mesh.getElements(dim=3)
n_nodes    = len(node_tags)
n_elements = sum(len(et) for et in elem_tags)
print(f"Nodes:    {n_nodes:,}")
print(f"Elements: {n_elements:,}")

# Export MSH
msh_path = meshes_dir / f"{PART_NAME}.msh"
gmsh.write(str(msh_path))
print(f"MSH written: {msh_path}")
gmsh.finalize()

# Convert to XDMF
msh = meshio.read(str(msh_path))
tet_block = next((b for b in msh.cells if b.type == "tetra"), None)
tri_block = next((b for b in msh.cells if b.type == "triangle"), None)

assert tet_block is not None, (
    "No tetrahedral elements in mesh — volume meshing failed. "
    "Check that the STEP file contains a closed solid body."
)

xdmf_path = meshes_dir / f"{PART_NAME}.xdmf"
meshio.write(str(xdmf_path), meshio.Mesh(
    points=msh.points,
    cells={"tetra": tet_block.data},
    cell_data={"gmsh:physical": [
        msh.cell_data_dict["gmsh:physical"].get(
            "tetra", [1] * len(tet_block.data)
        )
    ]},
))

bnd_path = meshes_dir / f"{PART_NAME}_boundaries.xdmf"
if tri_block is not None:
    meshio.write(str(bnd_path), meshio.Mesh(
        points=msh.points,
        cells={"triangle": tri_block.data},
        cell_data={"gmsh:physical": [
            msh.cell_data_dict["gmsh:physical"].get(
                "triangle", [1] * len(tri_block.data)
            )
        ]},
    ))

print(f"XDMF written: {xdmf_path}")
print(f"Boundaries:   {bnd_path}")

Cell 5 — Write stage02 handoff directly

In [ ]:
# Cell 5 — Write stage02 handoff
#
# STEP import skips Stage 1 AND Stage 2 of the normal pipeline —
# gmsh meshing happened in this notebook. We write a stage02 handoff
# so 03_fea_fenicsx.ipynb picks up seamlessly.

import json
from pathlib import Path
from datetime import datetime
from src.meshing.mesh_quality import check_mesh_quality, QualityThresholds

# Run quality check
print("Checking mesh quality...")
quality = check_mesh_quality(msh_path)
print(quality.summary())

handoff = {
    "stage":           "02_meshing",
    "origin":          "imported_step",
    "source_file":     str(step_input.resolve()),
    "part_name":       PART_NAME,
    "stl_path":        None,            # no STL in STEP import path
    "msh_path":        str(msh_path),
    "xdmf_path":       str(xdmf_path),
    "xdmf_boundaries": str(bnd_path),
    "n_nodes":         n_nodes,
    "n_elements":      n_elements,
    "quality": {
        "aspect_ratio_mean": quality.aspect_ratio_mean,
        "aspect_ratio_max":  quality.aspect_ratio_max,
        "aspect_ratio_p95":  quality.aspect_ratio_p95,
        "min_dihedral_deg":  quality.min_dihedral_deg,
        "n_inverted":        quality.n_inverted,
        "passed":            quality.passed,
    },
    "load_hints": {
        "primary_face":     PRIMARY_FACE,
        "load_magnitude_n": LOAD_MAGNITUDE_N,
    },
    "imported_at": datetime.utcnow().isoformat() + "Z",
}

handoff_path = Path(OUTPUT_DIR) / "meshes" / f"{PART_NAME}_stage02.json"
handoff_path.write_text(json.dumps(handoff, indent=2))

print(f"\n✅ Ready for Stage 3.")
print(f"   Open 03_fea_fenicsx.ipynb and run all cells.")
print(f"   Handoff: {handoff_path}")

Where this fits in the overall architecture
User has:         Enters at:                    Skips:
─────────────────────────────────────────────────────────────
params.json       01_geometry_openscad.ipynb    nothing
Own STL           00_import_stl.ipynb           Stage 1
Own STEP          00_import_step.ipynb          Stage 1 + Stage 2
The STEP path is the most powerful — a user with a STEP file from Fusion 360, SolidWorks, or FreeCAD gets a better mesh than even the OpenSCAD path, because gmsh meshes exact CAD surfaces rather than triangulated approximations. This is worth highlighting in the README as the preferred import path for users with existing CAD work.
The one thing to add to README.md under a new Bringing your own geometry section would be a simple decision tree:
Do you have an existing CAD file?
├── Yes, STEP/STP → use 00_import_step.ipynb  (best mesh quality)
├── Yes, STL      → use 00_import_stl.ipynb   (good, slightly lower quality)
└── No            → edit scad/params.json and use the standard pipeline